# AN Price Forecaster — Data Exploration

Run this after `run_ingestion.py` to verify data quality and explore correlations.

Run from the project root: `jupyter notebook` or open in VS Code.

In [ ]:
import sys
sys.path.append('..')  # Allow imports from project root

import pandas as pd
import matplotlib.pyplot as plt

from utils.db import read_table

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

## 1. Check data availability across all sources

In [ ]:
tables = ['fred_raw', 'yfinance_raw', 'gie_raw', 'worldbank_raw', 'ahdb_raw', 'ember_raw']

for table in tables:
    try:
        df = read_table(table)
        print(f"{table}: {len(df):,} rows | {df['data_date'].min()} to {df['data_date'].max()}")
    except Exception as e:
        print(f"{table}: ERROR — {e}")

## 2. UK AN spot price (target variable)

In [ ]:
ahdb = read_table('ahdb_raw')
ahdb['data_date'] = pd.to_datetime(ahdb['data_date'])
ahdb = ahdb.sort_values('data_date')

print(ahdb.describe())

plt.figure(figsize=(14, 4))
plt.plot(ahdb['data_date'], ahdb['price_gbp_t'])
plt.title('UK AN Spot Price (GBP/tonne) — AHDB')
plt.xlabel('Date')
plt.ylabel('GBP/tonne')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. FRED series — gas, FX, Brent

In [ ]:
fred = read_table('fred_raw')
fred['data_date'] = pd.to_datetime(fred['data_date'])

print('FRED series available:')
print(fred.groupby('series_name').agg({'data_date': ['min', 'max', 'count']}))

# Plot each series
for series_name, group in fred.groupby('series_name'):
    plt.figure(figsize=(14, 3))
    plt.plot(group['data_date'], group['value'])
    plt.title(f'FRED: {series_name} ({group["series_id"].iloc[0]})')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## 4. EU gas storage (GIE AGSI+)

In [ ]:
gie = read_table('gie_raw')
gie['data_date'] = pd.to_datetime(gie['data_date'])
gie = gie.sort_values('data_date')

fig, axes = plt.subplots(2, 1, figsize=(14, 6))
axes[0].plot(gie['data_date'], gie['full_pct'])
axes[0].set_title('EU Gas Storage — % Full (GIE AGSI+)')
axes[0].set_ylabel('% Full')
axes[0].grid(True, alpha=0.3)

axes[1].plot(gie['data_date'], gie['gas_in_storage'])
axes[1].set_title('EU Gas Storage — Volume (TWh)')
axes[1].set_ylabel('TWh')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. World Bank commodity prices

In [ ]:
wb = read_table('worldbank_raw')
wb['data_date'] = pd.to_datetime(wb['data_date'])

for commodity, group in wb.groupby('commodity'):
    group = group.sort_values('data_date')
    plt.figure(figsize=(14, 3))
    plt.plot(group['data_date'], group['value'])
    plt.title(f'World Bank: {commodity} (USD/tonne)')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## 6. Quick correlation check (AN price vs key drivers)

Align all series to monthly frequency and compute Pearson correlations with UK AN price.
This is an early-stage sense check — not a feature selection exercise.

In [ ]:
# Resample everything to monthly
def to_monthly(df, date_col, value_col, name):
    df = df.set_index(date_col)[value_col].resample('MS').mean()
    return df.rename(name)

an = to_monthly(ahdb.set_index('data_date').reset_index(), 'data_date', 'price_gbp_t', 'AN_GBP_t')

fred_pivot = fred.pivot_table(index='data_date', columns='series_name', values='value')
fred_monthly = fred_pivot.resample('MS').mean()

gie_monthly = to_monthly(gie.set_index('data_date').reset_index(), 'data_date', 'full_pct', 'eu_gas_storage_pct')

combined = pd.concat([an, fred_monthly, gie_monthly], axis=1).dropna()

if not combined.empty:
    print('Pearson correlations with UK AN price:')
    print(combined.corr()['AN_GBP_t'].sort_values(ascending=False).to_string())
else:
    print('Not enough overlapping data for correlations — check data coverage.')